In [ ]:
!pip -q install -U transformers datasets accelerate peft evaluate jiwer soundfile librosa openai-whisper

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
import whisper, torchaudio
import os, glob, torch, librosa
from datasets import Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate
import numpy as np

In [ ]:
#################################
#       LOAD SPEAKERS           #
#################################

from google.colab import drive
drive.mount('/content/drive')

import os
from datasets import Dataset

AUDIO_ROOT = "/content/drive/MyDrive/REAL DATA/dataset/impaired"
TEXT_ROOT  = "/content/drive/MyDrive/REAL DATA/dataset/transcripts"

TRAIN_N = 12

train_data, test_data = [], []

speakers = sorted([
    s for s in os.listdir(AUDIO_ROOT)
    if os.path.isdir(os.path.join(AUDIO_ROOT, s))
])

for idx, speaker in enumerate(speakers):
    speaker_path = os.path.join(AUDIO_ROOT, speaker)
    target = train_data if idx < TRAIN_N else test_data

    for i in range(1, 71):
        wav = f"{i:02d}.wav"
        txt = f"{i:02d}.txt"

        wav_path = os.path.join(speaker_path, wav)
        txt_path = os.path.join(TEXT_ROOT, txt)

        if not os.path.exists(wav_path) or not os.path.exists(txt_path):
            continue

        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip()

        target.append({"audio": wav_path, "text": text, "speaker": speaker})

train_ds = Dataset.from_list(train_data)
test_ds  = Dataset.from_list(test_data)

print("Speakers:", len(speakers))
print("Train samples:", len(train_ds))
print("Test samples:", len(test_ds))
print("Example:", train_ds[0])


In [ ]:
#################################
#   load Whisper + add LoRA     #
#################################

import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name, language="tl", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(DEVICE)

# Force Tagalog decoding prompt
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="tl", task="transcribe")

# LoRA
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj","k_proj","out_proj"]
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


In [ ]:
# Preprocessing
import librosa

def preprocess(batch):
    audio, sr = librosa.load(batch["audio"], sr=16000)
    batch["input_features"] = processor.feature_extractor(
        audio, sampling_rate=16000
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_ds_proc = train_ds.map(preprocess, remove_columns=train_ds.column_names)
test_ds_proc  = test_ds.map(preprocess, remove_columns=test_ds.column_names)

print(train_ds_proc[0].keys())



In [ ]:
# Save to Drive
SAVE_ROOT = "/content/drive/MyDrive/whisper_ds_cached"

train_ds_proc.save_to_disk(SAVE_ROOT + "/train")
test_ds_proc.save_to_disk(SAVE_ROOT + "/test")

print("Saved to:", SAVE_ROOT)

In [ ]:
#Load preprocessed ds from Drive
from datasets import load_from_disk

SAVE_ROOT = "/content/drive/MyDrive/whisper_ds_cached"

train_ds_proc = load_from_disk(SAVE_ROOT + "/train")
test_ds_proc  = load_from_disk(SAVE_ROOT + "/test")

print("Loaded:", len(train_ds_proc), len(test_ds_proc))

In [ ]:
import evaluate
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

wer = evaluate.load("wer")

class DataCollatorWhisper:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer.compute(predictions=pred_str, references=label_str)}

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper_tl_impaired_lora",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=50,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    generation_max_length=128,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_proc,
    eval_dataset=test_ds_proc,
    data_collator=DataCollatorWhisper(processor),
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
save_dir = "/content/drive/MyDrive/whisper_tl_impaired_lora_adapter"
model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)
print("Saved:", save_dir)

In [ ]:
#load model
from google.colab import drive
drive.mount('/content/drive')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL = "openai/whisper-small"
ADAPTER_DIR = "/content/drive/MyDrive/whisper_tl_impaired_lora_adapter"  # change if needed

processor = WhisperProcessor.from_pretrained(ADAPTER_DIR)
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)

base_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="tl", task="transcribe"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).to(DEVICE)
model.eval()

print("Model loaded.")


In [ ]:
import torch, librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL = "openai/whisper-small"
ADAPTER_DIR = "/content/drive/MyDrive/whisper_tl_impaired_lora_adapter"

processor = WhisperProcessor.from_pretrained(ADAPTER_DIR)

base = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL).to(DEVICE)
base.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="tl", task="transcribe")

model = PeftModel.from_pretrained(base, ADAPTER_DIR).to(DEVICE)
model.eval()

wav_path = "/content/drive/MyDrive/REAL DATA/dataset/impaired/speaker_15/59.wav"

audio, sr = librosa.load(wav_path, sr=16000)

# Whisper expects log-mel features shaped [batch, 80, frames]
features = processor.feature_extractor(audio, sampling_rate=16000).input_features
features = torch.tensor(features).to(DEVICE)

with torch.no_grad():
    predicted_ids = model.generate(
        input_features=features,   # ✅ Whisper-specific arg name
        max_new_tokens=256,
        num_beams=1
    )

text = processor.tokenizer.decode(predicted_ids[0], skip_special_tokens=True)

print("TRANSCRIPTION:")
print(text)
